# 🧠 Construa sua própria Rede Neural Convolucional em 3, 2, 1...!

**Aula prática de Redes Neurais Convolucionais (CNN) com Python e TensorFlow**

> *Esta aula faz parte da disciplina de Teoria dos Grafos. Uma Rede Neural é, fundamentalmente, um grafo dirigido — e você vai ver isso ao vivo!*

---

## 📑 Sumário

1. [Por que Python? Por que TensorFlow?](#python-tensorflow)
2. [A Rede Neural como Grafo](#grafo)
3. [O que é o MNIST?](#mnist)
4. [Carregando e Explorando os Dados](#explorando)
5. [Pré-processamento](#preprocessamento)
6. [Arquitetura CNN — Construindo na Mão](#arquitetura)
7. [Visualizando a Camada FC como Grafo (2D e 3D)](#fc-grafo)
8. [Treinamento](#treinamento)
9. [Métricas de Validação Completas](#metricas)
10. [Transfer Learning — VGG16, ResNet e MobileNet](#transfer)
11. [Testando com Outro Dataset — CIFAR-10](#cifar)
12. [Avanços Pós-CNN](#avancos)
13. [Conclusão](#conclusao)

---
## 1. Por que Python? Por que TensorFlow? <a id='python-tensorflow'></a>

### 🐍 Por que Python?

Python é a **língua franca** do Aprendizado de Máquina e do Aprendizado Profundo:

| Motivo | Detalhe |
|---|---|
| **Sintaxe simples** | Curva de aprendizado suave, ideal para prototipagem rápida |
| **Ecossistema rico** | NumPy, Pandas, Matplotlib, Scikit-learn, TensorFlow, PyTorch… |
| **Comunidade gigante** | Milhões de usuários, fóruns ativos, Stack Overflow |
| **Interoperabilidade** | Integra com C/C++, Java, R, MATLAB |
| **Suporte nativo a GPU** | Via CUDA e cuDNN através das bibliotecas de DL |

---

### ⚡ Por que TensorFlow?

[TensorFlow](https://www.tensorflow.org/) é um framework de código aberto desenvolvido pelo **Google Brain Team** (2015):

- **Grafos de computação** estáticos e dinâmicos (Eager Execution desde TF 2.0)
- **API Keras** integrada — construção intuitiva de redes
- Suporte a **CPU, GPU e TPU**
- Deploy via TF Serving, TF Lite (mobile) e TF.js (web)

#### 🌍 Com quais linguagens o TensorFlow é compatível?

| Linguagem | Suporte |
|---|---|
| **Python** | Principal (suporte completo) |
| **C++** | API nativa de baixo nível |
| **JavaScript** | TensorFlow.js — roda no navegador |
| **Java** | API oficial |
| **Go** | API experimental |
| **Swift** | Swift for TensorFlow |
| **R** | Via pacote `tensorflow` no CRAN |

---

### 🔄 Alternativas ao TensorFlow

| Framework | Mantido por | Destaque |
|---|---|---|
| **PyTorch** | Meta | Gráfos dinâmicos, favorito na pesquisa acadêmica |
| **Keras** | Google | API de alto nível; hoje integrada ao TF |
| **Theano** | MILA/Montreal | Pioneiro histórico (descontinuado em 2017) |
| **MXNet** | Apache/Amazon | Usado na AWS SageMaker |
| **JAX** | Google | Diferenciação automática + XLA |
| **Caffe/Caffe2** | Berkeley/Meta | CNNs eficientes para visão computacional |
| **PaddlePaddle** | Baidu | Popular na China |

> **Nota:** Usaremos **TensorFlow 2.x com Keras** — combinação mais usada em cursos introdutórios.

In [ ]:
# Instalação (no Google Colab, descomente se necessário)
# !pip install tensorflow matplotlib seaborn scikit-learn networkx plotly

import tensorflow as tf
import numpy as np
import matplotlib
import sklearn
import networkx as nx
import plotly

print(f"TensorFlow  : {tf.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Matplotlib  : {matplotlib.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"NetworkX    : {nx.__version__}")
print(f"Plotly      : {plotly.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n🚀 GPU disponível: {gpus}")
else:
    print("\n⚠️  Nenhuma GPU detectada. Usando CPU.")

---
## 2. A Rede Neural como Grafo <a id='grafo'></a>

### 🔗 Conexão com a Teoria dos Grafos

Na disciplina de **Teoria dos Grafos**, estudamos que um **grafo** $G = (V, E)$ é composto por:
- $V$ — conjunto de **vértices** (nós)
- $E$ — conjunto de **arestas** (conexões entre nós)

Uma **Rede Neural Artificial** é exatamente um **dígrafo ponderado** (grafo dirigido com pesos):

| Conceito de Grafo | Correspondência na RNA |
|---|---|
| Vértice (nó) | Neurônio artificial |
| Aresta dirigida | Conexão sináptica |
| Peso da aresta | Peso sináptico $w_{ij}$ |
| Caminho | Propagação do sinal (forward pass) |
| Ciclo | Ausente em redes feedforward (é um DAG) |

### 📐 Propriedades do Dígrafo de uma CNN

- É um **DAG (Grafo Acíclico Dirigido)** na fase de inferência
- O **grau de entrada** de cada neurônio FC = número de neurônios da camada anterior
- O **grau de saída** = número de neurônios da camada seguinte
- Total de pesos FC = $m \times n$ (grafo bipartido completo $K_{m,n}$)

> 💡 **Exemplo:** Se uma camada tem 100 neurônios e a seguinte tem 50 → $100 \times 50 = 5.000$ pesos!

### 🔬 Do Neurônio Biológico ao Modelo Computacional

A inspiração para as Redes Neurais Artificiais vem do funcionamento do **cérebro humano**:

![O Neurônio Biológico](imagens/O-Neuronio-Biologico.png)

Um neurônio biológico recebe sinais pelos **dendritos**, integra-os no **corpo celular (soma)** e, se o sinal ultrapassar um limiar, dispara um **potencial de ação** pelo **axônio** em direção a outros neurônios.

### 🤖 O Modelo de McCulloch-Pitts (1943)

Warren McCulloch e Walter Pitts formalizaram matematicamente esse comportamento:

![O Modelo de McCulloch-Pitts](imagens/O-modelo-de-McCulloch-Pitts.png)

$$y = f\left(\sum_{i=1}^{n} w_i \cdot x_i + b\right)$$

| Componente Biológico | Equivalente Computacional |
|---|---|
| Dendrito | Entrada $x_i$ |
| Sinapse | Peso $w_i$ |
| Soma (corpo celular) | $\sum w_i x_i + b$ |
| Limiar de disparo | Função de ativação $f$ |
| Axônio | Saída $y$ |

### 🏗️ Evolução das Arquiteturas de Redes Neurais

**Single Layer Perceptron** — a rede mais simples: apenas entrada e saída direta.

![Arquitetura Single Layer Perceptron](imagens/Arquitetura-do-modelo-Single-Layer-Perceptron.png)

**Limitação:** resolve apenas problemas **linearmente separáveis**:

| Linearmente Separável ✅ | Não Linearmente Separável ❌ |
|:---:|:---:|
| ![Linearmente Separável](imagens/Linearmente-Separavel.png) | ![Não Linearmente Separável](imagens/Nao-Linearmente-Separavel.png) |

**Multilayer Perceptron (MLP)** — adiciona camadas ocultas para resolver problemas não-lineares:

![Arquitetura Multilayer Perceptron](imagens/Arquitetura-do-modelo-Multilayer-Perceptron.png)

> 💡 A **CNN** é uma evolução do MLP especializada para **dados com estrutura espacial** (imagens). Em vez de conectar cada pixel a todos os neurônios, ela aplica **filtros convolucionais** que detectam padrões locais de forma eficiente.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def draw_neural_net_graph(layer_sizes, layer_labels=None,
                           title="Rede Neural como Grafo Dirigido"):
    G = nx.DiGraph()
    pos = {}
    node_colors = []
    color_map = ['#4FC3F7', '#81C784', '#FFB74D', '#E57373', '#CE93D8']

    node_id = 0
    layer_nodes = []
    for l_idx, size in enumerate(layer_sizes):
        nodes_in_layer = []
        for n_idx in range(size):
            G.add_node(node_id)
            pos[node_id] = (l_idx * 3, -(n_idx - size / 2))
            node_colors.append(color_map[l_idx % len(color_map)])
            nodes_in_layer.append(node_id)
            node_id += 1
        layer_nodes.append(nodes_in_layer)

    for l in range(len(layer_nodes) - 1):
        for src in layer_nodes[l]:
            for dst in layer_nodes[l + 1]:
                G.add_edge(src, dst)

    fig, ax = plt.subplots(figsize=(14, 6))
    nx.draw(G, pos, ax=ax, node_color=node_colors, with_labels=False,
            node_size=600, arrows=True, arrowsize=8,
            edge_color='#B0BEC5', alpha=0.85, width=0.5)

    if layer_labels:
        for l_idx, label in enumerate(layer_labels):
            ax.text(l_idx * 3, max(layer_sizes) / 2 + 0.8, label,
                    ha='center', fontsize=10, fontweight='bold', color='#37474F')

    patches = [mpatches.Patch(color=color_map[i % len(color_map)],
               label=layer_labels[i] if layer_labels else f"Camada {i+1}")
               for i in range(len(layer_sizes))]
    ax.legend(handles=patches, loc='lower right', fontsize=9)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    print(f"Nós (neurônios): {G.number_of_nodes()} | Arestas (pesos): {G.number_of_edges()}")

draw_neural_net_graph(
    layer_sizes=[4, 6, 6, 3],
    layer_labels=['Entrada (4)', 'Oculta 1 (6)', 'Oculta 2 (6)', 'Saída (3)'],
    title="MLP como Grafo Acíclico Dirigido (DAG)"
)

---
## 3. O que é o MNIST? <a id='mnist'></a>

### 📦 MNIST — O "Hello World" do Aprendizado Profundo

O **MNIST** (*Modified National Institute of Standards and Technology*) é o dataset mais famoso de Visão Computacional:

| Característica | Detalhe |
|---|---|
| **Criado por** | Yann LeCun, Corinna Cortes, Christopher Burges |
| **Ano** | 1998 |
| **Conteúdo** | Dígitos manuscritos de 0 a 9 |
| **Total** | 70.000 imagens |
| **Treino / Teste** | 60.000 / 10.000 |
| **Resolução** | 28 × 28 pixels |
| **Canal** | Escala de cinza (1 canal) |
| **Classes** | 10 (dígitos 0–9) |

### 🏛️ Contexto Histórico

Construído a partir de dois bancos de dados do NIST:
- **SD-1**: imagens escritas por estudantes do ensino médio
- **SD-3**: imagens escritas por funcionários do Census Bureau (EUA)

### 🔗 Conexão com Grafos

Cada imagem percorre um **grafo computacional**:

```
Imagem (28×28) → Conv → ReLU → Pooling → Conv → ReLU → Pooling → Flatten → FC → Softmax → Classe
```

No TensorFlow, cada operação é um **nó do grafo computacional**!

### 🖼️ Como o Computador "Enxerga" as Imagens?

Imagens digitais são armazenadas como **matrizes de números inteiros**. Cada posição da matriz corresponde a um **pixel** com um valor entre 0 (preto) e 255 (branco):

![Matriz de valores de pixel](imagens/Matriz-de-valores-de-pixel.png)

Ao aplicar zoom extremo em qualquer imagem digital, os pixels individuais ficam visíveis:

![Apple Zoom](imagens/Apple-Zoom.png)

- Uma imagem MNIST de 28×28 = **784 pixels** no total.
- O computador processa uma **matriz de 784 números**, não uma imagem visual.

### 🎨 Imagens em Escala de Cinza vs. Coloridas (RGB)

![RGB](imagens/RGB-2.png)

| Tipo | Canais | Valores por pixel | Exemplo |
|---|---|---|---|
| Escala de cinza | 1 | 0–255 | MNIST |
| RGB colorida | 3 (R, G, B) | 3 × (0–255) | CIFAR-10, fotos reais |

O MNIST usa escala de cinza (1 canal) → formato `(28, 28, 1)` no TensorFlow.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Baixa automaticamente (ou usa cache) as 70.000 imagens do MNIST
# Formato: arrays NumPy de uint8 com valores entre 0 e 255
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

print("=== Dataset MNIST ===")
print(f"Treino  → X: {X_train.shape} | y: {y_train.shape}")  # (60000, 28, 28)
print(f"Teste   → X: {X_test.shape}  | y: {y_test.shape}")   # (10000, 28, 28)
print(f"Tipo    → {X_train.dtype}")    # uint8 (inteiro sem sinal de 8 bits)
print(f"Range   → {X_train.min()} – {X_train.max()}")  # 0 a 255
print(f"Classes → {np.unique(y_train)}")  # [0 1 2 3 4 5 6 7 8 9]

---
## 4. Carregando e Explorando os Dados <a id='explorando'></a>

Antes de qualquer modelagem, precisamos **conhecer os dados**:
- Visualizar exemplos de cada classe
- Analisar a distribuição de classes (dataset balanceado?)
- Entender como o computador "enxerga" as imagens (matriz numérica)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')

# ── 1. Exemplos de cada dígito ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(18, 4))
fig.suptitle('MNIST — Um exemplo de cada classe (treino / teste)',
             fontsize=13, fontweight='bold')
for digit in range(10):
    i_tr = np.where(y_train == digit)[0][0]
    i_te = np.where(y_test  == digit)[0][0]
    axes[0, digit].imshow(X_train[i_tr], cmap='gray')
    axes[0, digit].set_title(f'Treino\n"{digit}"', fontsize=8)
    axes[0, digit].axis('off')
    axes[1, digit].imshow(X_test[i_te], cmap='gray')
    axes[1, digit].set_title(f'Teste\n"{digit}"', fontsize=8)
    axes[1, digit].axis('off')
plt.tight_layout(); plt.show()

# ── 2. Distribuição de classes ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, X, y, title in [(axes[0], X_train, y_train, 'Treino'),
                         (axes[1], X_test,  y_test,  'Teste')]:
    unique, counts = np.unique(y, return_counts=True)
    bars = ax.bar(unique, counts, color=plt.cm.tab10.colors)
    ax.set_title(f'Distribuição de Classes — {title}', fontweight='bold')
    ax.set_xlabel('Dígito'); ax.set_ylabel('Quantidade')
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(cnt), ha='center', fontsize=8)
plt.tight_layout(); plt.show()

# ── 3. Como o computador enxerga a imagem ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sample = X_train[0]
axes[0].imshow(sample, cmap='gray')
axes[0].set_title(f'Dígito "{y_train[0]}" — 28×28 pixels', fontweight='bold')
axes[0].axis('off')
im = axes[1].imshow(sample, cmap='Blues')
for i in range(0, 28, 4):
    for j in range(0, 28, 4):
        axes[1].text(j, i, str(sample[i, j]), ha='center', va='center',
                     fontsize=5, color='red' if sample[i,j]>127 else 'black')
axes[1].set_title('Valores dos pixels (subamostrado)', fontweight='bold')
plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout(); plt.show()

---
## 5. Pré-processamento dos Dados <a id='preprocessamento'></a>

### Por que normalizar?
Pixels no intervalo $[0, 255]$ → normalizar para $[0, 1]$:
- Facilita a **convergência** do gradiente descendente
- Melhora estabilidade numérica
- Funções de ativação (ReLU, Softmax) funcionam melhor

### Por que adicionar dimensão de canal?
TensorFlow/Keras espera formato `(batch, altura, largura, canais)`.  
MNIST escala de cinza: 1 canal → `(N, 28, 28, 1)`

### One-Hot Encoding
Softmax produz 10 probabilidades. Os rótulos devem estar em OHE:  
$7 \rightarrow [0,0,0,0,0,0,0,\mathbf{1},0,0]$

In [ ]:
from tensorflow.keras.utils import to_categorical

NUM_CLASSES = 10  # Dígitos 0–9

# ── 1. Normalização ───────────────────────────────────────────────────────────
# Converte de uint8 [0, 255] para float32 [0.0, 1.0]
# Isso acelera a convergência e estabiliza os gradientes
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm  = X_test.astype('float32')  / 255.0

# ── 2. Adição da dimensão de canal (escala de cinza = 1 canal) ────────────────
# TensorFlow/Keras espera o formato (batch, altura, largura, canais)
# np.newaxis adiciona um eixo extra: (60000, 28, 28) → (60000, 28, 28, 1)
X_train_norm = X_train_norm[..., np.newaxis]  # (60000, 28, 28, 1)
X_test_norm  = X_test_norm[..., np.newaxis]   # (10000, 28, 28, 1)

# ── 3. One-Hot Encoding dos rótulos ──────────────────────────────────────────
# Softmax produz 10 probabilidades; os rótulos devem ter o mesmo formato
# Exemplo: 7 → [0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
y_train_ohe = to_categorical(y_train, NUM_CLASSES)
y_test_ohe  = to_categorical(y_test,  NUM_CLASSES)

print("=== Após pré-processamento ===")
print(f"X_train : {X_train_norm.shape} | range: [{X_train_norm.min():.1f}, {X_train_norm.max():.1f}]")
print(f"X_test  : {X_test_norm.shape}")
print(f"y_train OHE — dígito '{y_train[0]}': {y_train_ohe[0]}")

---
## 6. Arquitetura CNN — Construindo na Mão <a id='arquitetura'></a>

### 🏗️ Fluxo da Arquitetura

```
INPUT (28×28×1)
    │
    ▼
CONV1: 32 filtros (3×3), ReLU, BatchNorm  →  (26×26×32)
    │
    ▼
CONV2: 64 filtros (3×3), ReLU, BatchNorm  →  (24×24×64)
    │
    ▼
MAX POOL 2×2, stride=2                    →  (12×12×64)
    │
    ▼
DROPOUT 25%
    │
    ▼
FLATTEN                                   →  (9216,)
    │
    ▼
DENSE: 128 neurônios, ReLU               →  (128,)
    │
    ▼
DROPOUT 50%
    │
    ▼
DENSE: 10, Softmax                        →  (10,)
```

### 📐 Parâmetros que você controla

| Parâmetro | Valor | Significado |
|---|---|---|
| `filters` | 32, 64 | Número de filtros convolucionais |
| `kernel_size` | (3,3) | Tamanho da janela de convolução |
| `activation` | 'relu' | $f(x)=\max(0,x)$ |
| `pool_size` | (2,2) | Janela de Max-Pooling |
| `dropout` | 0.25, 0.50 | Regularização — "desliga" neurônios aleatoriamente |
| `dense_units` | 128 | Neurônios na camada FC |
| `optimizer` | Adam | Taxa de aprendizado adaptativa |
| `loss` | categorical_crossentropy | Função de perda multiclasse |

### 🗺️ Visão Geral da Arquitetura CNN

![Arquitetura do Modelo CNN](imagens/Arquitetura-do-modelo-CNN.png)

A seguir, vamos detalhar **cada etapa** do fluxo:
```
Imagem (28×28) → Conv → ReLU → Pooling → Conv → ReLU → Pooling → Flatten → FC → Softmax → Classe
```

### 🖼️ Etapa 1 — Imagem de Entrada (28×28×1)

A entrada da rede é uma imagem 28×28 em escala de cinza, representada como uma **matriz de valores normalizados** entre 0.0 e 1.0:

![Matriz de valores de pixel](imagens/Matriz-de-valores-de-pixel.png)

Adicionamos uma **dimensão de canal** para compatibilidade com o TensorFlow:
- Formato original: `(28, 28)` → Formato com canal: `(28, 28, 1)`
- Em um batch de 128 imagens: `(128, 28, 28, 1)`

### 🔲 Etapa 2 — Convolução (Conv)

A operação de **convolução** desliza um **filtro** (kernel) de tamanho 3×3 sobre a imagem inteira. Em cada posição, realiza uma multiplicação elementar e soma os resultados, gerando um **feature map** (mapa de características):

$$\text{saída}[i,j] = \sum_{m=0}^{2} \sum_{n=0}^{2} \text{imagem}[i+m,\, j+n] \times \text{filtro}[m,n]$$

**Exemplo visual — a imagem original, a janela de recorte (receptive field) e o resultado após convolução:**

| Imagem Original | Área do Filtro (Receptive Field) | Resultado (Convoluído) |
|:---:|:---:|:---:|
| ![Museu Original](imagens/Museu.png) | ![Área Recortada](imagens/Area-recortada.png) | ![Museu Convoluído](imagens/Museu-convolved.png) |

**Na nossa CNN:**
- **CONV1**: 32 filtros 3×3, `padding='valid'` → (28×28) torna-se (26×26) → shape: `(26×26×32)`
- **CONV2**: 64 filtros 3×3, `padding='valid'` → (26×26) torna-se (24×24) → shape: `(24×24×64)`

> 💡 Cada filtro aprende a detectar um **padrão diferente**: bordas horizontais, verticais, curvas, cantos, etc.

### ⚡ Etapa 3 — Ativação ReLU (Rectified Linear Unit)

Após cada convolução, aplicamos a função de ativação **ReLU**:

$$\text{ReLU}(x) = \max(0,\, x)$$

| Valor de entrada | Saída ReLU |
|---|---|
| Negativo (ex.: −3.5) | **0** (neurônio "desligado") |
| Zero | **0** |
| Positivo (ex.: 4.2) | **4.2** (mantido intacto) |

**Por que é necessária?**
- Sem não-linearidade, empilhar camadas equivale a **uma única transformação linear** — independente do número de camadas.
- ReLU é computacionalmente eficiente e evita o problema de **vanishing gradient** melhor que Sigmoid/Tanh.
- Após CONV + ReLU, somamos **BatchNormalization** para estabilizar a distribuição das ativações durante o treino.

### 🏊 Etapa 4 — Max Pooling (Subamostragem)

O **Max Pooling** divide o feature map em regiões de 2×2 e mantém apenas o **valor máximo** de cada região:

![Max Pooling](imagens/Max-Pooling.png)

**Efeito na dimensionalidade:**

```
Antes do Pooling : (24 × 24 × 64) = 36.864 valores
Depois do Pooling: (12 × 12 × 64) = 9.216 valores  ← redução de 75%!
```

**Benefícios:**
- 📉 **Reduz dimensionalidade**: menos parâmetros nas próximas camadas
- 🔄 **Invariância a pequenas translações**: um padrão detectado levemente deslocado ainda é reconhecido
- 🛡️ **Reduz overfitting**: a rede foca nos padrões mais fortes

> 💡 `pool_size=(2,2)` com `stride=2` (padrão) corta a resolução espacial **pela metade** em cada dimensão.

### 📐 Etapa 5 — Flatten (Achatar)

O **Flatten** converte o tensor 3D em um **vetor 1D** contíguo:

```
(12 × 12 × 64) = 9.216 valores  →  [x₁, x₂, x₃, …, x₉₂₁₆]
```

Isso é necessário porque as camadas **Dense (totalmente conectadas)** esperam entrada unidimensional. Não há parâmetros aprendidos nesta etapa — é apenas uma **reestruturação** dos dados.

### 🧠 Etapa 6 — Camada Totalmente Conectada (FC) + Softmax

**Camada Dense/FC (128 neurônios):**

Cada um dos 9.216 valores conecta-se a **todos os 128 neurônios** — é o grafo bipartido completo $K_{9216,\,128}$:

$$\text{total de pesos} = 9.216 \times 128 = \mathbf{1.179.648 \text{ pesos!}}$$

Após Dropout de 50% (regularização), passamos à camada de saída.

---

**Camada de Saída — Softmax (10 neurônios):**

Produz uma **distribuição de probabilidades** sobre as 10 classes de dígitos:

$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\displaystyle\sum_{j=0}^{9} e^{z_j}}, \quad \sum_{i=0}^{9} P(\text{classe}_i) = 1$$

**Exemplo de saída para a imagem do dígito "7":**

| Classe | 0 | 1 | 2 | 3 | 4 | 5 | 6 | **7** | 8 | 9 |
|--------|---|---|---|---|---|---|---|-------|---|---|
| Prob.  | 0.00 | 0.01 | 0.00 | 0.00 | 0.01 | 0.00 | 0.00 | **0.97** | 0.00 | 0.01 |

O modelo prediz a classe com a **maior probabilidade** → Dígito **7** ✅

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
    BatchNormalization, Input
)
from tensorflow.keras.optimizers import Adam

# ── Hiperparâmetros da arquitetura ────────────────────────────────────────────
INPUT_SHAPE   = (28, 28, 1)   # Altura × Largura × Canais
CONV1_FILTERS = 32            # Número de filtros na 1ª camada Conv
CONV2_FILTERS = 64            # Número de filtros na 2ª camada Conv (mais complexidade)
KERNEL_SIZE   = (3, 3)        # Tamanho de cada filtro convolucional
POOL_SIZE     = (2, 2)        # Janela do Max Pooling (reduz dimensão pela metade)
DENSE_UNITS   = 128           # Neurônios na camada totalmente conectada
DROPOUT_1     = 0.25          # 25% de dropout após o Pooling (regularização leve)
DROPOUT_2     = 0.50          # 50% de dropout antes da saída (regularização forte)
LEARNING_RATE = 1e-3          # Taxa de aprendizado inicial do Adam

model = Sequential([
    Input(shape=INPUT_SHAPE),   # Define a forma da entrada explicitamente

    # ── Bloco Convolucional 1 ─────────────────────────────────────────────────
    # 32 filtros 3×3 detectam padrões simples (bordas, curvas)
    # padding='valid' → sem preenchimento; saída (26×26×32)
    Conv2D(CONV1_FILTERS, KERNEL_SIZE, activation='relu',
           padding='valid', name='Conv1'),
    BatchNormalization(name='BN1'),  # Normaliza ativações → treino mais estável

    # ── Bloco Convolucional 2 ─────────────────────────────────────────────────
    # 64 filtros 3×3 detectam padrões mais complexos (partes de dígitos)
    # saída: (24×24×64)
    Conv2D(CONV2_FILTERS, KERNEL_SIZE, activation='relu',
           padding='valid', name='Conv2'),
    BatchNormalization(name='BN2'),

    # ── Subamostragem (Max Pooling) ───────────────────────────────────────────
    # Mantém apenas o valor máximo em cada janela 2×2
    # saída: (12×12×64) — redução de 75% no número de valores!
    MaxPooling2D(pool_size=POOL_SIZE, name='MaxPool1'),
    Dropout(DROPOUT_1, name='Drop1'),  # Desliga 25% dos neurônios aleatoriamente

    # ── Flatten: tensor 3D → vetor 1D ────────────────────────────────────────
    # (12 × 12 × 64) = 9.216 valores em um vetor único
    Flatten(name='Flatten'),

    # ── Camada Totalmente Conectada (FC) ──────────────────────────────────────
    # Cada um dos 9.216 valores se conecta a 128 neurônios = 1.179.648 pesos!
    Dense(DENSE_UNITS, activation='relu', name='FC1'),
    Dropout(DROPOUT_2, name='Drop2'),  # Desliga 50% para evitar overfitting

    # ── Camada de Saída ────────────────────────────────────────────────────────
    # Softmax normaliza os 10 valores em probabilidades (soma = 1.0)
    Dense(NUM_CLASSES, activation='softmax', name='Output')
], name='CNN_MNIST')

# Compilação: define otimizador, função de perda e métrica
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),  # Adam: taxa adaptativa por parâmetro
    loss='categorical_crossentropy',              # Perda padrão para classificação multiclasse
    metrics=['accuracy']                          # Monitora acurácia durante o treino
)

model.summary()  # Exibe tabela com camadas, shapes e número de parâmetros

---
## 7. Visualizando a Camada FC como Grafo <a id='fc-grafo'></a>

### 🕸️ A Camada FC é um Grafo Bipartido Completo $K_{m,n}$!

Em Teoria dos Grafos, o **grafo bipartido completo** $K_{m,n}$ possui:
- $m$ vértices no conjunto A, $n$ no conjunto B
- Toda aresta de A conecta a **todos** os vértices de B
- Total de arestas: $m \times n$

Na nossa CNN:
- **Flatten** produz $12 \times 12 \times 64 = 9.216$ entradas
- **FC** tem 128 neurônios
- Total de pesos: $9.216 \times 128 = \mathbf{1.179.648}$ conexões!

Vamos visualizar uma **versão reduzida** em 2D e 3D interativo.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Usamos versões reduzidas (12 e 8) para visualização clara
# Na CNN real seria K(9216, 128) — impossível de visualizar!
N_IN  = 12   # representa os neurônios da camada Flatten
N_OUT = 8    # representa os neurônios da camada FC

# Cria grafo bipartido completo K(N_IN, N_OUT)
B = nx.complete_bipartite_graph(N_IN, N_OUT)
pos = nx.bipartite_layout(B, nodes=range(N_IN))  # layout em dois lados

# Separa os nós por conjunto para colorir diferente
input_nodes  = list(range(N_IN))
output_nodes = list(range(N_IN, N_IN + N_OUT))

fig, ax = plt.subplots(figsize=(13, 6))
nx.draw_networkx_nodes(B, pos, nodelist=input_nodes,  node_color='#4FC3F7',
                       node_size=700, ax=ax, label='Flatten (entrada)')
nx.draw_networkx_nodes(B, pos, nodelist=output_nodes, node_color='#81C784',
                       node_size=700, ax=ax, label='FC Neurônios (saída)')
# alpha=0.15 torna as arestas semitransparentes → evita poluição visual
nx.draw_networkx_edges(B, pos, ax=ax, alpha=0.15, edge_color='#90A4AE', width=0.8)
nx.draw_networkx_labels(B, pos,
    labels={n: f'x{n}' for n in input_nodes}, font_size=8, ax=ax)
nx.draw_networkx_labels(B, pos,
    labels={n: f'h{n-N_IN}' for n in output_nodes}, font_size=8, ax=ax)

ax.set_title(
    f'Grafo Bipartido Completo K({N_IN},{N_OUT}) — Camada FC (versão reduzida)\n'
    f'Real seria K(9216,128) = 1.179.648 pesos!',
    fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.axis('off')
plt.tight_layout()
plt.show()
print(f"K({N_IN},{N_OUT}): {B.number_of_nodes()} nós, {B.number_of_edges()} arestas")
print(f"K(9216,128) real:  9344 nós, {9216*128:,} arestas")

In [ ]:
import plotly.graph_objects as go
import numpy as np

N_IN  = 10
N_OUT = 6
np.random.seed(42)

in_pos  = np.column_stack([np.zeros(N_IN),
                            np.linspace(-1, 1, N_IN),
                            np.linspace(-1, 1, N_IN)])
out_pos = np.column_stack([np.ones(N_OUT),
                            np.linspace(-0.8, 0.8, N_OUT),
                            np.linspace(-0.8, 0.8, N_OUT)])

edge_x, edge_y, edge_z = [], [], []
for i in range(N_IN):
    for j in range(N_OUT):
        edge_x += [in_pos[i,0], out_pos[j,0], None]
        edge_y += [in_pos[i,1], out_pos[j,1], None]
        edge_z += [in_pos[i,2], out_pos[j,2], None]

fig3d = go.Figure(data=[
    go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines',
                 line=dict(color='rgba(150,150,150,0.3)', width=1),
                 name='Pesos (arestas)'),
    go.Scatter3d(x=in_pos[:,0], y=in_pos[:,1], z=in_pos[:,2],
                 mode='markers+text',
                 marker=dict(size=10, color='#4FC3F7', opacity=0.9),
                 text=[f'x{i}' for i in range(N_IN)],
                 textposition='middle left',
                 name='Flatten (entrada)'),
    go.Scatter3d(x=out_pos[:,0], y=out_pos[:,1], z=out_pos[:,2],
                 mode='markers+text',
                 marker=dict(size=12, color='#81C784', opacity=0.9, symbol='diamond'),
                 text=[f'h{j}' for j in range(N_OUT)],
                 textposition='middle right',
                 name='FC Neurônios')
])

fig3d.update_layout(
    title=dict(
        text=f'Camada FC como Grafo 3D — K({N_IN},{N_OUT})<br>'
             '<sup>Cada aresta = um peso sináptico w_ij</sup>',
        font=dict(size=14)
    ),
    scene=dict(
        xaxis_title='Camada', yaxis_title='Posição Y', zaxis_title='Posição Z',
        bgcolor='rgb(240,240,250)'
    ),
    legend=dict(x=0.02, y=0.98),
    margin=dict(l=0, r=0, b=0, t=60),
    height=550
)
fig3d.show()

print(f"\n💡 Discussão em sala:")
print(f"  • K({N_IN},{N_OUT}): {N_IN*N_OUT} pesos para aprender")
print(f"  • K(9216,128) real: {9216*128:,} pesos!")
print(f"  • Dropout 'desliga' conexões aleatoriamente → como remover arestas do grafo!")

---
## 8. Treinamento <a id='treinamento'></a>

### ⚙️ Callbacks de Treinamento

| Callback | Função |
|---|---|
| **EarlyStopping** | Para o treino se a validação parar de melhorar |
| **ReduceLROnPlateau** | Reduz taxa de aprendizado quando a loss estagna |
| **ModelCheckpoint** | Salva o melhor modelo automaticamente |

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# ── Hiperparâmetros de treinamento ────────────────────────────────────────────
BATCH_SIZE = 128   # Número de amostras por atualização de pesos (mini-batch)
EPOCHS     = 20    # Número máximo de épocas (Early Stopping pode interromper antes)
VAL_SPLIT  = 0.15  # 15% do treino reservado para validação (~9.000 imagens)

# ── Callbacks ─────────────────────────────────────────────────────────────────
callbacks = [
    # Para o treinamento se val_loss não melhorar por 5 épocas consecutivas
    # restore_best_weights=True → reverte aos pesos da melhor época automaticamente
    EarlyStopping(monitor='val_loss', patience=5,
                  restore_best_weights=True, verbose=1),

    # Reduz a taxa de aprendizado à metade se val_loss não melhorar por 3 épocas
    # min_lr=1e-6 → não deixa a LR cair abaixo desse limite
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-6, verbose=1),

    # Salva em disco apenas o modelo com melhor val_accuracy
    ModelCheckpoint('melhor_modelo_cnn.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

print("Iniciando treinamento...")
history = model.fit(
    X_train_norm, y_train_ohe,   # Dados de treino (normalizados + one-hot)
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=VAL_SPLIT,  # Separa 15% para validação a cada época
    callbacks=callbacks,
    verbose=1
)
print("\n✅ Treinamento concluído!")

---
## 9. Métricas de Validação Completas <a id='metricas'></a>

| Métrica | Descrição |
|---|---|
| **Acurácia** | Proporção de predições corretas |
| **Precisão** | TP / (TP + FP) |
| **Recall** | TP / (TP + FN) |
| **F1-Score** | Média harmônica entre Precisão e Recall |
| **Matriz de Confusão** | Tabela predições × rótulos reais |
| **ROC / AUC** | Curva ROC e área sob a curva por classe |
| **Curvas de Loss/Acc** | Evolução durante o treinamento |
| **Análise qualitativa** | Exemplos corretos e erros do modelo |
| **Feature Maps** | O que cada filtro "enxerga" |

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Treino',    color='#1565C0', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validação', color='#C62828', linewidth=2, linestyle='--')
axes[0].set_title('Curva de Loss (Cross-Entropy)', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=11); axes[0].grid(True, alpha=0.4)

axes[1].plot(history.history['accuracy'],     label='Treino',    color='#2E7D32', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validação', color='#E65100', linewidth=2, linestyle='--')
axes[1].set_title('Curva de Acurácia', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Acurácia')
axes[1].legend(fontsize=11); axes[1].grid(True, alpha=0.4)

plt.suptitle('Histórico de Treinamento — CNN MNIST',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)
import seaborn as sns
import numpy as np

# ── Gerar predições no conjunto de teste ──────────────────────────────────────
# predict() retorna probabilidades Softmax para cada uma das 10 classes
y_pred_proba = model.predict(X_test_norm, verbose=0)

# argmax pega o índice da maior probabilidade → classe predita
y_pred = np.argmax(y_pred_proba, axis=1)  # shape: (10000,)
y_true = y_test                            # rótulos reais (não OHE)

# ── Métricas escalares ────────────────────────────────────────────────────────
# Acurácia: (predições corretas) / (total de amostras)
acc  = accuracy_score(y_true, y_pred)

# Precisão: para cada classe, TP / (TP + FP)  — "quando diz X, está certo?"
prec = precision_score(y_true, y_pred, average='weighted')

# Recall: para cada classe, TP / (TP + FN)  — "de todos os X reais, quantos achou?"
rec  = recall_score(y_true, y_pred, average='weighted')

# F1-Score: média harmônica entre Precisão e Recall
f1   = f1_score(y_true, y_pred, average='weighted')

print("=" * 50)
print("    MÉTRICAS GERAIS (conjunto de teste)")
print("=" * 50)
print(f"  Acurácia            : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precisão (weighted) : {prec:.4f}")
print(f"  Recall (weighted)   : {rec:.4f}")
print(f"  F1-Score (weighted) : {f1:.4f}")
print("=" * 50)
print()
# Relatório por classe: precisão, recall e F1 individuais para cada dígito
print(classification_report(y_true, y_pred,
      target_names=[f'Dígito {i}' for i in range(10)]))

In [ ]:
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=range(10), yticklabels=range(10))
axes[0].set_title('Matriz de Confusão (absoluta)', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Predito'); axes[0].set_ylabel('Real')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1],
            xticklabels=range(10), yticklabels=range(10))
axes[1].set_title('Matriz de Confusão (normalizada)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Predito'); axes[1].set_ylabel('Real')

plt.suptitle('Matrizes de Confusão — CNN MNIST', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nErros por classe:")
for i in range(10):
    errors = cm[i].sum() - cm[i, i]
    print(f"  Dígito {i}: {errors:4d} erros de {cm[i].sum():5d} ({errors/cm[i].sum()*100:.1f}%)")

In [ ]:
from sklearn.metrics import roc_curve, auc
from tensorflow.keras.utils import to_categorical

y_test_ohe_local = to_categorical(y_true, NUM_CLASSES)
fig, ax = plt.subplots(figsize=(10, 8))
colors  = plt.cm.tab10.colors
aucs    = []

for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_test_ohe_local[:, i], y_pred_proba[:, i])
    roc_auc = auc(fpr, tpr)
    aucs.append(roc_auc)
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'Dígito {i} (AUC = {roc_auc:.4f})')

ax.plot([0,1],[0,1],'k--', linewidth=1.5, label='Aleatório')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('Taxa de Falsos Positivos (FPR)', fontsize=12)
ax.set_ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=12)
ax.set_title('Curvas ROC — One-vs-Rest\nCNN MNIST', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
print(f"\nAUC médio (macro): {np.mean(aucs):.4f}")

In [ ]:
correct_idx = np.where(y_pred == y_true)[0]
wrong_idx   = np.where(y_pred != y_true)[0]

fig, axes = plt.subplots(2, 10, figsize=(18, 5))
fig.suptitle('Análise Qualitativa — Predições Corretas (verde) e Erradas (vermelho)',
             fontsize=12, fontweight='bold')

for i in range(10):
    idx = correct_idx[i]
    axes[0, i].imshow(X_test_norm[idx, :, :, 0], cmap='gray')
    conf = y_pred_proba[idx, y_pred[idx]]
    axes[0, i].set_title(f'✅ {y_pred[idx]}\n({conf:.0%})', fontsize=8, color='green')
    axes[0, i].axis('off')

    idx = wrong_idx[i]
    axes[1, i].imshow(X_test_norm[idx, :, :, 0], cmap='Reds')
    conf = y_pred_proba[idx, y_pred[idx]]
    axes[1, i].set_title(f'❌ R:{y_true[idx]}→P:{y_pred[idx]}\n({conf:.0%})',
                         fontsize=7, color='red')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()
print(f"\nTotal de erros: {len(wrong_idx)} de {len(y_true)} ({len(wrong_idx)/len(y_true)*100:.2f}%)")

In [ ]:
import tensorflow as tf

layer_names   = ['Conv1', 'Conv2', 'MaxPool1']
feature_model = tf.keras.Model(
    inputs=model.input,
    outputs=[model.get_layer(n).output for n in layer_names]
)

sample_img   = X_test_norm[0:1]
feature_maps = feature_model.predict(sample_img, verbose=0)

for layer_name, fmap in zip(layer_names, feature_maps):
    n_filters = min(fmap.shape[-1], 16)
    fig, axes = plt.subplots(2, n_filters // 2, figsize=(16, 3))
    fig.suptitle(f'Feature Maps — {layer_name} '
                 f'({fmap.shape[1]}×{fmap.shape[2]}×{fmap.shape[3]})',
                 fontsize=11, fontweight='bold')
    for k, ax in enumerate(axes.flat):
        if k < n_filters:
            ax.imshow(fmap[0, :, :, k], cmap='viridis')
            ax.set_title(f'Filtro {k}', fontsize=7)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

---
## 10. Transfer Learning — VGG16, ResNet e MobileNet <a id='transfer'></a>

### 🔄 O que é Transfer Learning?

**Transfer Learning** reutiliza um modelo pré-treinado em um grande dataset (ImageNet, 1.4M imagens, 1.000 classes) e o adapta para um novo problema.

**Analogia com grafos:** reutilizamos o **dígrafo de pesos** já otimizado e trocamos apenas os **nós de saída**!

### 🏛️ Modelos Famosos

| Modelo | Ano | Profundidade | Top-5 Acc | Parâmetros |
|---|---|---|---|---|
| AlexNet | 2012 | 8 | 84.6% | ~60M |
| **VGG16** | 2014 | 16 | 92.7% | ~138M |
| GoogLeNet | 2014 | 22 | 93.3% | ~7M |
| **ResNet50** | 2015 | 50 | 95.3% | ~25M |
| **MobileNetV2** | 2018 | 54 | 91.0% | ~3.4M |
| EfficientNetB0 | 2019 | — | 93.3% | ~5.3M |

> **Por que VGG16 primeiro?** Arquitetura sequencial clara — perfeita para visualizar como grafo!

In [ ]:
from tensorflow.keras.applications import VGG16, MobileNetV2
from tensorflow.keras import Model
from tensorflow.keras.layers import (GlobalAveragePooling2D, Dense,
                                     Dropout, Input, Resizing)
import tensorflow as tf

def build_transfer_model(base_fn, input_shape=(32, 32, 3),
                          num_classes=10, model_name='TransferModel'):
    inp = Input(shape=(28, 28, 1), name='mnist_input')
    x   = Resizing(input_shape[0], input_shape[1], name='resize')(inp)
    x   = tf.keras.layers.Lambda(
              lambda t: tf.repeat(t, 3, axis=-1), name='gray2rgb')(x)

    base = base_fn(weights='imagenet', include_top=False,
                   input_shape=input_shape)
    base.trainable = False   # congelamos pesos pré-treinados

    x   = base(x, training=False)
    x   = GlobalAveragePooling2D(name='GAP')(x)
    x   = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax', name='output')(x)

    m = Model(inputs=inp, outputs=out, name=model_name)
    m.compile(optimizer=Adam(learning_rate=1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
    return m, base

# ── VGG16 ─────────────────────────────────────────────────────────────────────
print("Carregando VGG16...")
model_vgg, base_vgg = build_transfer_model(VGG16, model_name='VGG16_MNIST')

print(f"\nParâmetros congelados (VGG16): {base_vgg.count_params():,}")
print(f"Parâmetros treináveis         : {model_vgg.trainable_count:,}")
model_vgg.summary()

In [ ]:
print("Treinando VGG16...")
history_vgg = model_vgg.fit(
    X_train_norm, y_train_ohe,
    batch_size=256, epochs=10,
    validation_split=0.15,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3,
                             restore_best_weights=True)],
    verbose=1
)
_, acc_vgg = model_vgg.evaluate(X_test_norm, y_test_ohe, verbose=0)
print(f"\n✅ VGG16 — Acurácia no Teste: {acc_vgg*100:.2f}%")

# ── MobileNetV2 ───────────────────────────────────────────────────────────────
print("\nCarregando MobileNetV2...")
model_mob, _ = build_transfer_model(MobileNetV2, input_shape=(32, 32, 3),
                                    model_name='MobileNetV2_MNIST')
history_mob = model_mob.fit(
    X_train_norm, y_train_ohe,
    batch_size=256, epochs=10,
    validation_split=0.15,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3,
                             restore_best_weights=True)],
    verbose=1
)
_, acc_mob = model_mob.evaluate(X_test_norm, y_test_ohe, verbose=0)
print(f"✅ MobileNetV2 — Acurácia no Teste: {acc_mob*100:.2f}%")

In [ ]:
_, acc_cnn = model.evaluate(X_test_norm, y_test_ohe, verbose=0)

model_names = ['CNN do Zero', 'VGG16\n(Transfer)', 'MobileNetV2\n(Transfer)']
acuracias   = [acc_cnn * 100, acc_vgg * 100, acc_mob * 100]
cores_bar   = ['#1565C0', '#6A1B9A', '#1B5E20']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(model_names, acuracias, color=cores_bar, edgecolor='white', linewidth=1.5)
ax.bar_label(bars, fmt='%.2f%%', fontsize=12, fontweight='bold', padding=3)
ax.set_ylim([min(acuracias) - 2, 101])
ax.set_ylabel('Acurácia no Teste (%)', fontsize=12)
ax.set_title('Comparação de Modelos — MNIST', fontsize=13, fontweight='bold')
ax.axhline(y=99, color='gray', linestyle='--', alpha=0.5, label='99% referência')
ax.legend(fontsize=10); ax.grid(True, alpha=0.4, axis='y')
plt.tight_layout()
plt.show()

---
## 11. Testando com Outro Dataset — CIFAR-10 <a id='cifar'></a>

### 🎨 MNIST vs CIFAR-10

| Característica | MNIST | CIFAR-10 |
|---|---|---|
| Imagens | 70.000 | 60.000 |
| Resolução | 28×28 | 32×32 |
| Canais | 1 (cinza) | 3 (RGB) |
| Classes | 10 dígitos | 10 objetos reais |
| Dificuldade | 🟢 Fácil | 🔴 Difícil |

**Classes do CIFAR-10:** avião, automóvel, pássaro, gato, cervo, cachorro, sapo, cavalo, navio, caminhão

Veremos como a **mesma abordagem** performa num dataset mais desafiador, motivando o uso de modelos mais profundos (ResNet) e **Data Augmentation**.

In [ ]:
(X_c_train, y_c_train), (X_c_test, y_c_test) = tf.keras.datasets.cifar10.load_data()

CIFAR_CLASSES = ['Avião','Automóvel','Pássaro','Gato','Cervo',
                 'Cachorro','Sapo','Cavalo','Navio','Caminhão']

print(f"X_train: {X_c_train.shape} | X_test: {X_c_test.shape}")

fig, axes = plt.subplots(2, 10, figsize=(18, 4))
fig.suptitle('CIFAR-10 — Exemplos por Classe', fontsize=13, fontweight='bold')
for cls_idx in range(10):
    for row, offset in enumerate([0, 1]):
        idx = np.where(y_c_train.flatten() == cls_idx)[0][offset]
        axes[row, cls_idx].imshow(X_c_train[idx])
        if row == 0:
            axes[row, cls_idx].set_title(CIFAR_CLASSES[cls_idx], fontsize=8)
        axes[row, cls_idx].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

X_c_train_n   = X_c_train.astype('float32') / 255.0
X_c_test_n    = X_c_test.astype('float32')  / 255.0
y_c_train_flat = y_c_train.flatten()
y_c_test_flat  = y_c_test.flatten()
y_c_train_ohe  = to_categorical(y_c_train_flat, 10)
y_c_test_ohe   = to_categorical(y_c_test_flat,  10)

# Data Augmentation
datagen = ImageDataGenerator(rotation_range=15, width_shift_range=0.1,
                              height_shift_range=0.1, horizontal_flip=True,
                              zoom_range=0.1)
datagen.fit(X_c_train_n)

# CNN para CIFAR-10 (mais profunda)
model_cifar = Sequential([
    Input(shape=(32, 32, 3)),
    Conv2D(32, (3,3), activation='relu', padding='same'), BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)), Dropout(0.25),

    Conv2D(64, (3,3), activation='relu', padding='same'), BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)), Dropout(0.25),

    Conv2D(128, (3,3), activation='relu', padding='same'), BatchNormalization(),
    MaxPooling2D((2,2)), Dropout(0.35),

    Flatten(),
    Dense(256, activation='relu'), Dropout(0.5),
    Dense(10, activation='softmax')
], name='CNN_CIFAR10')

model_cifar.compile(optimizer=Adam(learning_rate=1e-3),
                    loss='categorical_crossentropy', metrics=['accuracy'])
model_cifar.summary()

In [ ]:
CIFAR_EPOCHS = 30
CIFAR_BATCH  = 128

history_cifar = model_cifar.fit(
    datagen.flow(X_c_train_n, y_c_train_ohe, batch_size=CIFAR_BATCH),
    steps_per_epoch=len(X_c_train_n) // CIFAR_BATCH,
    epochs=CIFAR_EPOCHS,
    validation_data=(X_c_test_n, y_c_test_ohe),
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=7,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=4, min_lr=1e-6, verbose=1)
    ],
    verbose=1
)
_, acc_c = model_cifar.evaluate(X_c_test_n, y_c_test_ohe, verbose=0)
print(f"\n✅ CNN CIFAR-10 — Acurácia no Teste: {acc_c*100:.2f}%")

In [ ]:
y_c_pred = np.argmax(model_cifar.predict(X_c_test_n, verbose=0), axis=1)

# Curvas de treinamento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_cifar.history['loss'],     linewidth=2, label='Treino')
axes[0].plot(history_cifar.history['val_loss'], linewidth=2, linestyle='--', label='Validação')
axes[0].set_title('Loss — CNN CIFAR-10', fontweight='bold')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.4)

axes[1].plot(history_cifar.history['accuracy'],     linewidth=2, label='Treino')
axes[1].plot(history_cifar.history['val_accuracy'], linewidth=2, linestyle='--', label='Validação')
axes[1].set_title('Acurácia — CNN CIFAR-10', fontweight='bold')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Acurácia')
axes[1].legend(); axes[1].grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

# Matriz de confusão
cm_c = confusion_matrix(y_c_test_flat, y_c_pred)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_c, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=CIFAR_CLASSES, yticklabels=CIFAR_CLASSES)
ax.set_title('Matriz de Confusão — CNN CIFAR-10', fontweight='bold', fontsize=13)
ax.set_xlabel('Predito'); ax.set_ylabel('Real')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

print(classification_report(y_c_test_flat, y_c_pred, target_names=CIFAR_CLASSES))

---
## 12. Avanços Pós-CNN: O que veio depois? <a id='avancos'></a>

### 🚀 Linha do Tempo da Visão Computacional

```
1998  ── LeNet-5 (LeCun) ──────── Primeiras CNNs práticas
2012  ── AlexNet ─────────────── Revolução ImageNet (GPU + Dropout + ReLU)
2014  ── VGGNet / GoogLeNet ──── Redes mais profundas
2015  ── ResNet ──────────────── Conexões Residuais (152 camadas!)
2016  ── DenseNet ───────────── Conexões densas entre camadas
2017  ── Transformers ─────────── Mecanismo de Atenção (NLP)
2018  ── EfficientNet ─────────── Escalamento composto
2020  ── Vision Transformer ─── Transformers para Visão!
2021  ── CLIP / DALL-E ─────────── Visão + Linguagem
2022  ── Stable Diffusion ──────── Geração por difusão
2023+ ── GPT-4V / Gemini ─────── Modelos Multimodais Gigantes
```

### 📌 Principais Inovações

#### 1. Conexões Residuais — ResNet (2015)
- **Problema:** Redes profundas sofrem *vanishing gradient*
- **Solução:** Skip connections: $F(x) + x$
- **No grafo:** cria **caminhos paralelos** no DAG!

#### 2. Transformers — Atenção (2017)
- Cada elemento "presta atenção" em todos os outros → matriz de atenção
- **No grafo:** grafo completo ponderado onde todos os nós se conectam!

#### 3. Vision Transformer — ViT (2020)
- Imagem dividida em **patches** tratados como tokens de texto
- Competitivo com CNNs sem nenhuma convolução

#### 4. Modelos de Difusão (2022)
- Adição e remoção gradual de ruído (processo markoviano)
- Base do Stable Diffusion, DALL-E 2, Midjourney

#### 5. Modelos Multimodais (2023+)
- **CLIP**: alinha texto e imagem num espaço latente compartilhado
- **GPT-4V, Gemini, LLaVA**: compreendem imagens e texto juntos

### 🔮 Tendências

| Área | Tendência |
|---|---|
| **Eficiência** | TinyML, quantização, modelos compactos |
| **Multimodalidade** | Visão + texto + áudio + vídeo |
| **Self-supervised** | Aprender sem rótulos (DINO, MAE, SimCLR) |
| **Graph Neural Networks** | CNNs generalizadas para **grafos arbitrários**! |

> **Conexão final:** As **Graph Neural Networks (GNNs)** operando em grafos moleculares, redes sociais e redes de conhecimento são a fronteira atual — e são a generalização natural de tudo que aprendemos!

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

modelos_timeline = [
    (1998, 'LeNet-5',           7.0,  '#78909C'),
    (2012, 'AlexNet',          60.0,  '#1565C0'),
    (2014, 'VGG16',           138.0,  '#6A1B9A'),
    (2014, 'GoogLeNet',         7.0,  '#AD1457'),
    (2015, 'ResNet-50',        25.0,  '#00838F'),
    (2017, 'DenseNet-201',     20.0,  '#2E7D32'),
    (2019, 'EfficientNet-B7',  66.0,  '#E65100'),
    (2020, 'ViT-L/16',        307.0,  '#B71C1C'),
    (2021, 'CLIP (ViT-L)',    428.0,  '#4527A0'),
    (2023, 'GPT-4V (est.)',  1760.0,  '#1A237E'),
]

anos    = [m[0] for m in modelos_timeline]
nomes   = [m[1] for m in modelos_timeline]
params  = [m[2] for m in modelos_timeline]
cores_t = [m[3] for m in modelos_timeline]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(anos, params, s=[p/3 + 40 for p in params],
                c=cores_t, zorder=5, alpha=0.9)
for i, (a, n, p) in enumerate(zip(anos, nomes, params)):
    axes[0].annotate(n, (a, p), textcoords="offset points",
                     xytext=(5, 5 if i%2==0 else -15),
                     fontsize=8, color=cores_t[i], fontweight='bold')
axes[0].set_yscale('log')
axes[0].set_xlabel('Ano', fontsize=12)
axes[0].set_ylabel('Parâmetros (M) — escala log', fontsize=12)
axes[0].set_title('Evolução dos Modelos de Visão', fontweight='bold', fontsize=11)
axes[0].grid(True, alpha=0.4)

y_pos = np.arange(len(nomes))
axes[1].barh(y_pos, params, color=cores_t, edgecolor='white', linewidth=0.5)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels([f'{n} ({a})' for n, a in zip(nomes, anos)])
axes[1].set_xlabel('Parâmetros (M)', fontsize=12)
axes[1].set_title('Parâmetros por Modelo', fontweight='bold', fontsize=11)
axes[1].set_xscale('log')
axes[1].grid(True, alpha=0.4, axis='x')
for i, (v, c) in enumerate(zip(params, cores_t)):
    axes[1].text(v * 1.1, i, f'{v:.0f}M', va='center',
                 fontsize=8, color=c, fontweight='bold')

plt.suptitle('25 anos de evolução em Visão Computacional',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 13. Conclusão <a id='conclusao'></a>

### 🎯 Resumo

| Etapa | O que fizemos |
|---|---|
| **Teoria** | Fundamentamos CNN dentro da Teoria dos Grafos |
| **Dados** | Exploração e pré-processamento de MNIST e CIFAR-10 |
| **Construção** | CNN do zero, camada por camada, parâmetros manuais |
| **Visualização** | Camada FC como grafo bipartido 2D e 3D interativo |
| **Feature Maps** | O que cada filtro "enxerga" na imagem |
| **Treinamento** | Early Stopping, ReduceLR, ModelCheckpoint |
| **Métricas** | Acurácia, Precisão, Recall, F1, Confusão, ROC/AUC |
| **Transfer Learning** | VGG16 e MobileNetV2 adaptados |
| **CIFAR-10** | Data Augmentation e CNN mais profunda |
| **História** | 25 anos de evolução pós-CNN |

### 🔗 Conexão Final com Teoria dos Grafos

```
Grafo Simples          → Perceptron (1 neurônio)
DAG Sequencial         → MLP / CNN (feedforward)
DAG com Skip Edges     → ResNet (conexões residuais)
Grafo Completo (Kₙₙ)   → Attention Mechanism (Transformers)
Grafo Arbitrário       → Graph Neural Networks (GNNs)
```

### 📚 Próximos Passos

1. **Fine-tuning**: Descongelar camadas do VGG16 com LR baixo
2. **ResNet do zero**: Implementar blocos residuais manualmente
3. **Object Detection**: YOLO, SSD, Faster R-CNN
4. **Segmentação**: U-Net — CNN em forma de "U"!
5. **GNNs**: PyTorch Geometric para grafos moleculares

### 📖 Referências

- LeCun et al. (1998). *Gradient-based learning applied to document recognition.* Proceedings of the IEEE.
- Krizhevsky, Sutskever & Hinton (2012). *ImageNet Classification with Deep CNNs.* NeurIPS.
- Simonyan & Zisserman (2014). *Very Deep Convolutional Networks.* (VGG)
- He et al. (2015). *Deep Residual Learning for Image Recognition.* (ResNet)
- Dosovitskiy et al. (2020). *An Image is Worth 16x16 Words.* (ViT)
- Goodfellow, Bengio & Courville (2016). *Deep Learning.* MIT Press.
- https://www.tensorflow.org/ | https://keras.io/

---

*Tutorial desenvolvido para a disciplina de Teoria dos Grafos.*  
*"Uma Rede Neural é um Grafo — e agora você construiu um!"* 🧠🕸️